# 01 — Architecture of a sales intelligence agent

> **Applied Labs** · **Domain**: AG · **Difficulty**: Advanced · **Reading time**: ~40 min

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/labs/02_sales_intelligence/01_architecture.ipynb)

**Prerequisites**:
- [00_first_principles.ipynb](./00_first_principles.ipynb) — Outreach economics and personalization depth
- Familiarity with LLM prompting and embedding models

**What you will learn**:
- How to design a multi-stage sales intelligence pipeline
- Architecture for prospect research, lead scoring, and email generation
- Response classification using embeddings and few-shot examples
- State machine design for multi-touch email sequences

In [ ]:
# @title Setup — Run this cell first
# --- Google Colab Setup ---
!pip install -q "openai>=1.30.0" "chromadb>=0.5.0" "sentence-transformers>=2.7.0"

import os
import json
from dataclasses import dataclass, field
from typing import Optional
from enum import Enum
import numpy as np
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer("all-MiniLM-L6-v2")
print("Setup complete — embedding model loaded.")

## Section 1 — System overview

The sales intelligence agent is a pipeline of specialized components, each handling one stage of the outreach process:

```
┌─────────────┐    ┌──────────────┐    ┌─────────────┐    ┌──────────────┐
│  Prospect    │───▶│  Research    │───▶│   Lead      │───▶│    Email     │
│   List       │    │   Agent      │    │  Scorer     │    │  Generator   │
└─────────────┘    └──────────────┘    └─────────────┘    └──────┬───────┘
                                                                  │
┌─────────────┐    ┌──────────────┐    ┌─────────────┐           │
│  Analytics   │◀──│  Follow-up   │◀──│  Response   │◀──────────┘
│  Dashboard   │    │   Agent      │    │ Classifier  │    (send queue)
└─────────────┘    └──────────────┘    └─────────────┘
```

Each component has a clear contract: defined inputs, outputs, and failure modes. This modularity lets us improve any stage independently.

In [ ]:
# Define the data contracts between pipeline stages
from dataclasses import dataclass, field
from typing import Optional

@dataclass
class Prospect:
    """Input to the pipeline — a target company/person."""
    company: str
    contact_name: str
    contact_title: str
    industry: str
    company_size: int
    website: str = ""
    linkedin_url: str = ""

@dataclass
class ResearchBrief:
    """Output of the research agent — structured intelligence."""
    prospect: Prospect
    recent_news: list[str] = field(default_factory=list)
    job_postings: list[str] = field(default_factory=list)
    tech_stack: list[str] = field(default_factory=list)
    pain_signals: list[str] = field(default_factory=list)
    personalization_hooks: list[str] = field(default_factory=list)
    confidence: float = 0.0

@dataclass
class LeadScore:
    """Output of the lead scorer."""
    prospect: Prospect
    total_score: float
    intent_score: float
    fit_score: float
    engagement_score: float
    tier: str  # "hot", "warm", "cold"

@dataclass
class GeneratedEmail:
    """Output of the email generator."""
    prospect: Prospect
    subject: str
    body: str
    personalization_hooks_used: list[str]
    tone: str
    cta_type: str

@dataclass
class ClassifiedResponse:
    """Output of the response classifier."""
    category: str  # positive, objection_price, objection_timing, etc.
    confidence: float
    suggested_action: str

# Show the pipeline data flow
pipeline_stages = [
    ("Prospect", "Raw contact info"),
    ("ResearchBrief", "Enriched intelligence"),
    ("LeadScore", "Prioritization tier"),
    ("GeneratedEmail", "Personalized outreach"),
    ("ClassifiedResponse", "Response handling"),
]
print("Pipeline Data Flow:")
for i, (name, desc) in enumerate(pipeline_stages):
    arrow = " → " if i < len(pipeline_stages) - 1 else ""
    print(f"  [{name}] {desc}{arrow}")

## Section 2 — Prospect research agent

The research agent is the intelligence backbone. It gathers structured data about a prospect from multiple sources. In production, these would be API calls to LinkedIn, Crunchbase, etc. Here we design the schema and simulate the data sources.

**Data source priority** (by signal value):
1. Job postings — strongest intent signal ("they're building something")
2. Funding/news — timing signal ("they have budget")
3. Tech stack — fit signal ("they use tools we integrate with")
4. Company profile — baseline context

In [ ]:
# Simulated data sources for the research agent
SIMULATED_COMPANY_DATA = {
    "DataFlow Inc": {
        "description": "Series B data infrastructure startup, 200 employees",
        "recent_news": [
            "Raised $45M Series B led by Accel (2 weeks ago)",
            "Launched real-time streaming product (1 month ago)",
            "CEO keynote at Data Summit on scaling challenges",
        ],
        "job_postings": [
            "Senior DevOps Engineer — Kubernetes, CI/CD pipelines",
            "Staff Backend Engineer — Python, distributed systems",
            "Engineering Manager — platform team, 8+ direct reports",
        ],
        "tech_stack": ["Python", "Kubernetes", "PostgreSQL", "Kafka", "dbt"],
    },
    "HealthBridge": {
        "description": "Digital health platform, 500 employees, SOC2 certified",
        "recent_news": [
            "Partnership with Mayo Clinic announced (1 week ago)",
            "HIPAA compliance automation product launched",
        ],
        "job_postings": [
            "Security Engineer — compliance automation, SOC2",
            "Senior Full-Stack Engineer — React, Node.js, FHIR",
        ],
        "tech_stack": ["React", "Node.js", "AWS", "PostgreSQL", "FHIR"],
    },
    "RetailEdge": {
        "description": "AI-powered retail analytics, 80 employees",
        "recent_news": [
            "Series A extension of $12M (3 weeks ago)",
            "Expanded to European market",
        ],
        "job_postings": [
            "ML Engineer — recommendation systems, PyTorch",
            "Data Engineer — Spark, Airflow, data pipelines",
        ],
        "tech_stack": ["Python", "PyTorch", "Spark", "Airflow", "Snowflake"],
    },
}

def research_prospect(company_name: str) -> dict:
    """Simulate researching a prospect across multiple data sources."""
    data = SIMULATED_COMPANY_DATA.get(company_name, {})
    if not data:
        return {"error": f"No data found for {company_name}"}

    # Extract personalization hooks
    hooks = []
    for news in data.get("recent_news", [])[:2]:
        hooks.append(f"Recent: {news}")
    for job in data.get("job_postings", [])[:2]:
        hooks.append(f"Hiring: {job}")
    if data.get("tech_stack"):
        hooks.append(f"Tech stack includes: {', '.join(data['tech_stack'][:3])}")

    return {
        "company": company_name,
        "description": data.get("description", ""),
        "news": data.get("recent_news", []),
        "jobs": data.get("job_postings", []),
        "tech_stack": data.get("tech_stack", []),
        "hooks": hooks,
        "hook_count": len(hooks),
    }

# Research all prospects
for company in SIMULATED_COMPANY_DATA:
    brief = research_prospect(company)
    print(f"\n=== {company} ({brief['hook_count']} hooks) ===")
    for hook in brief["hooks"]:
        print(f"  • {hook}")

## Section 3 — Lead scoring model

The lead scorer takes research data and computes a priority tier. The architecture uses configurable weights so sales teams can tune scoring to their ICP (Ideal Customer Profile).

Key design decision: **scores are explainable**. Every score comes with a breakdown showing which signals contributed most. This builds trust with sales teams who need to understand why a lead is ranked high.

In [ ]:
@dataclass
class ICPProfile:
    """Ideal Customer Profile — defines what a good prospect looks like."""
    target_industries: list[str] = field(default_factory=list)
    min_company_size: int = 50
    max_company_size: int = 1000
    required_tech: list[str] = field(default_factory=list)
    intent_keywords: list[str] = field(default_factory=list)

    # Scoring weights
    intent_weight: float = 0.45
    fit_weight: float = 0.35
    engagement_weight: float = 0.20

class LeadScorer:
    """Configurable multi-signal lead scoring engine."""

    def __init__(self, icp: ICPProfile):
        self.icp = icp

    def score_intent(self, research: dict) -> tuple[float, list[str]]:
        """Score intent signals from research data."""
        score = 0.0
        reasons = []
        news = research.get("news", [])
        jobs = research.get("jobs", [])

        # Funding signal
        funding_keywords = ["raised", "series", "funding", "investment"]
        if any(kw in " ".join(news).lower() for kw in funding_keywords):
            score += 0.4
            reasons.append("Recent funding detected")

        # Hiring signal
        if len(jobs) >= 2:
            score += 0.3
            reasons.append(f"Active hiring ({len(jobs)} postings)")

        # Tech change signal
        for kw in self.icp.intent_keywords:
            if any(kw.lower() in j.lower() for j in jobs):
                score += 0.3
                reasons.append(f"Hiring for '{kw}' — strong intent")
                break

        return min(score, 1.0), reasons

    def score_fit(self, research: dict) -> tuple[float, list[str]]:
        """Score how well the prospect matches the ICP."""
        score = 0.0
        reasons = []
        tech = set(t.lower() for t in research.get("tech_stack", []))
        required = set(t.lower() for t in self.icp.required_tech)

        overlap = tech & required
        if overlap:
            tech_score = len(overlap) / max(len(required), 1)
            score += tech_score * 0.6
            reasons.append(f"Tech overlap: {', '.join(overlap)}")

        # Industry and size would be checked here (simplified)
        score += 0.4  # baseline fit assumed from prospect list
        reasons.append("Industry match (from prospect list)")

        return min(score, 1.0), reasons

    def score(self, research: dict) -> dict:
        """Compute full lead score with explanations."""
        intent, intent_reasons = self.score_intent(research)
        fit, fit_reasons = self.score_fit(research)
        engagement = 0.3  # placeholder — would come from CRM data

        total = (intent * self.icp.intent_weight
                 + fit * self.icp.fit_weight
                 + engagement * self.icp.engagement_weight)

        tier = "hot" if total >= 0.6 else "warm" if total >= 0.35 else "cold"
        return {
            "total": round(total, 3), "tier": tier,
            "intent": round(intent, 3), "fit": round(fit, 3),
            "engagement": round(engagement, 3),
            "reasons": intent_reasons + fit_reasons,
        }

# Define ICP and score prospects
icp = ICPProfile(
    target_industries=["data infrastructure", "devtools", "cloud"],
    min_company_size=50, max_company_size=500,
    required_tech=["Python", "Kubernetes", "PostgreSQL"],
    intent_keywords=["DevOps", "CI/CD", "platform"],
)
scorer = LeadScorer(icp)

print(f"{'Company':<18s} {'Score':>6s} {'Tier':<6s} {'Top reason'}")
print("-" * 65)
for company in SIMULATED_COMPANY_DATA:
    research = research_prospect(company)
    result = scorer.score(research)
    top_reason = result["reasons"][0] if result["reasons"] else "—"
    print(f"{company:<18s} {result['total']:>6.3f} {result['tier']:<6s} {top_reason}")

## Section 4 — Email generation architecture

The email generator transforms research + ICP + value proposition into personalized outreach. The architecture uses a **prompt template with slots** that get filled from the research brief.

Key design principles:
- **Hook first**: Open with something specific to the prospect (not about you)
- **Bridge**: Connect their situation to your value
- **CTA**: Low-commitment ask (share a resource, not "buy now")
- **Tone calibration**: Match the prospect's likely communication style

In [ ]:
# Email generation prompt template architecture
EMAIL_PROMPT_TEMPLATE = """You are an expert sales development representative.
Write a personalized cold email using the research below.

RESEARCH BRIEF:
- Company: {company}
- Description: {description}
- Recent news: {news}
- Job postings: {jobs}
- Tech stack: {tech_stack}
- Personalization hooks: {hooks}

VALUE PROPOSITION: {value_prop}

TONE: {tone}
CTA TYPE: {cta_type}

RULES:
1. Open with a specific observation about the prospect (not about us)
2. Keep under 150 words
3. One clear CTA — {cta_type}
4. No buzzwords: "synergy", "leverage", "game-changer"
5. Sound like a knowledgeable human, not a marketing bot

Write the email with Subject: line first, then the body."""

# CTA variations
CTA_TYPES = {
    "resource_share": "Offer to share a relevant case study or resource",
    "quick_question": "Ask a specific question about their situation",
    "soft_meeting": "Suggest a brief 15-min chat, easy to decline",
    "intro_offer": "Offer a free assessment or audit",
}

# Tone calibration based on prospect signals
TONE_MAP = {
    "startup_exec": "Direct, concise, founder-to-founder energy",
    "enterprise_vp": "Professional, data-driven, ROI-focused",
    "technical_lead": "Technical depth, specific tooling references",
    "default": "Friendly professional, conversational",
}

def select_tone(title: str) -> str:
    """Select email tone based on prospect's title."""
    title_lower = title.lower()
    if any(t in title_lower for t in ["cto", "founder", "ceo"]):
        return "startup_exec"
    if any(t in title_lower for t in ["vp", "director", "head"]):
        return "enterprise_vp"
    if any(t in title_lower for t in ["engineer", "architect", "developer"]):
        return "technical_lead"
    return "default"

def build_email_prompt(research: dict, value_prop: str,
                       contact_title: str, cta_type: str = "resource_share") -> str:
    """Assemble the full email generation prompt from research data."""
    tone_key = select_tone(contact_title)
    return EMAIL_PROMPT_TEMPLATE.format(
        company=research["company"],
        description=research.get("description", ""),
        news="\n".join(f"  - {n}" for n in research.get("news", [])),
        jobs="\n".join(f"  - {j}" for j in research.get("jobs", [])),
        tech_stack=", ".join(research.get("tech_stack", [])),
        hooks="\n".join(f"  - {h}" for h in research.get("hooks", [])),
        value_prop=value_prop,
        tone=TONE_MAP[tone_key],
        cta_type=CTA_TYPES[cta_type],
    )

# Build a prompt for DataFlow
research = research_prospect("DataFlow Inc")
prompt = build_email_prompt(
    research,
    value_prop="We help engineering teams reduce deployment cycle time by 60%",
    contact_title="VP of Engineering",
    cta_type="resource_share",
)
print("=== Generated prompt (first 600 chars) ===")
print(prompt[:600])
print(f"\n... ({len(prompt)} chars total)")
print(f"\nTone selected: {select_tone('VP of Engineering')} → {TONE_MAP[select_tone('VP of Engineering')]}")

## Section 5 — Response classification

After sending emails, we must classify responses to route them correctly. The classifier uses embedding similarity against labeled examples — a few-shot approach that doesn't require fine-tuning.

| Category | Example | Next action |
|----------|---------|-------------|
| positive | "Sure, let's set up a call" | Route to calendar booking |
| objection_price | "Too expensive for our budget" | Price objection handler |
| objection_timing | "Not the right time" | Nurture sequence |
| objection_authority | "I'm not the right person" | Ask for referral |
| not_interested | "Please remove me from your list" | Suppress from sequences |
| out_of_office | "I'm OOO until Jan 15" | Reschedule follow-up |
| bounce | "Delivery failed" | Mark invalid, skip |

In [ ]:
# Response classifier using embedding similarity
RESPONSE_EXAMPLES = {
    "positive": [
        "Sure, I'd be happy to chat. How about next Tuesday?",
        "This looks interesting — can you send more details?",
        "Let's set up a quick call. Here's my calendar link.",
        "I've been looking for something like this. Let's talk.",
    ],
    "objection_price": [
        "Thanks but this is outside our budget right now.",
        "We don't have the budget for new tools this quarter.",
        "How much does this cost? We're pretty constrained.",
    ],
    "objection_timing": [
        "Not a good time — we're in the middle of a migration.",
        "Circle back in Q3, we're heads down on a launch.",
        "Interesting but we just signed a contract with a competitor.",
    ],
    "objection_authority": [
        "I'm not the right person for this. Try our CTO.",
        "You should talk to our VP of Engineering about this.",
        "This isn't my area — forwarding to the team lead.",
    ],
    "not_interested": [
        "Not interested, please remove me from your list.",
        "We're not looking for this kind of solution.",
        "No thanks.",
    ],
    "out_of_office": [
        "I'm out of office until January 15th with limited email.",
        "OOO — returning February 1st. For urgent matters contact...",
    ],
}

class ResponseClassifier:
    """Classify email responses using embedding similarity."""

    def __init__(self, model: SentenceTransformer, examples: dict):
        self.model = model
        self.categories = {}
        for category, texts in examples.items():
            embeddings = model.encode(texts)
            self.categories[category] = {
                "texts": texts,
                "centroid": np.mean(embeddings, axis=0),
            }

    def classify(self, response_text: str) -> dict:
        """Classify a response by similarity to category centroids."""
        response_emb = self.model.encode([response_text])[0]
        scores = {}
        for cat, data in self.categories.items():
            cos_sim = np.dot(response_emb, data["centroid"]) / (
                np.linalg.norm(response_emb) * np.linalg.norm(data["centroid"]))
            scores[cat] = float(cos_sim)
        best = max(scores, key=scores.get)
        return {"category": best, "confidence": round(scores[best], 3),
                "all_scores": {k: round(v, 3) for k, v in sorted(
                    scores.items(), key=lambda x: -x[1])}}

classifier = ResponseClassifier(embed_model, RESPONSE_EXAMPLES)

# Test on new responses
test_responses = [
    "Looks great, let's hop on a call this week!",
    "We just renewed our contract, maybe next year.",
    "Please take me off your mailing list.",
    "I'm on PTO until the 20th.",
    "Our VP of Product would be better suited for this.",
]
print("Response Classification Results:\n")
for resp in test_responses:
    result = classifier.classify(resp)
    print(f"  "{resp[:55]}..."")
    print(f"    → {result['category']} (confidence: {result['confidence']})\n")

## Section 6 — Sequence management

Cold outreach rarely converts on the first email. A sequence is a multi-touch campaign that adapts based on prospect behavior. The state machine below governs transitions:

```
                    ┌────────── no open ──────────┐
                    │                              ▼
[Email 1] ──open──▶ [Wait 3d] ──no reply──▶ [Email 2] ──reply──▶ [Classify]
                                                   │                    │
                                              no reply             ┌────┴────┐
                                                   │               │         │
                                                   ▼            positive  objection
                                              [Email 3]           │         │
                                                   │              ▼         ▼
                                              no reply        [Book]   [Handle]
                                                   │
                                                   ▼
                                              [Archive]
```

In [ ]:
class SequenceState(Enum):
    PENDING = "pending"
    EMAIL_1_SENT = "email_1_sent"
    EMAIL_1_OPENED = "email_1_opened"
    EMAIL_2_SENT = "email_2_sent"
    REPLIED = "replied"
    MEETING_BOOKED = "meeting_booked"
    OBJECTION_HANDLED = "objection_handled"
    ARCHIVED = "archived"
    SUPPRESSED = "suppressed"

@dataclass
class SequenceEntry:
    """Tracks a prospect's position in an outreach sequence."""
    prospect_company: str
    state: SequenceState = SequenceState.PENDING
    emails_sent: int = 0
    max_emails: int = 3
    days_between: int = 3
    history: list[str] = field(default_factory=list)

    def transition(self, event: str) -> str:
        """Apply a state transition based on an event."""
        old_state = self.state
        transitions = {
            (SequenceState.PENDING, "send"): SequenceState.EMAIL_1_SENT,
            (SequenceState.EMAIL_1_SENT, "open"): SequenceState.EMAIL_1_OPENED,
            (SequenceState.EMAIL_1_SENT, "no_open"): SequenceState.EMAIL_2_SENT,
            (SequenceState.EMAIL_1_OPENED, "reply"): SequenceState.REPLIED,
            (SequenceState.EMAIL_1_OPENED, "no_reply"): SequenceState.EMAIL_2_SENT,
            (SequenceState.EMAIL_2_SENT, "reply"): SequenceState.REPLIED,
            (SequenceState.EMAIL_2_SENT, "no_reply"): SequenceState.ARCHIVED,
            (SequenceState.REPLIED, "positive"): SequenceState.MEETING_BOOKED,
            (SequenceState.REPLIED, "objection"): SequenceState.OBJECTION_HANDLED,
            (SequenceState.REPLIED, "not_interested"): SequenceState.SUPPRESSED,
        }
        key = (self.state, event)
        if key in transitions:
            self.state = transitions[key]
            self.history.append(f"{old_state.value} --{event}--> {self.state.value}")
            if "send" in event or "sent" in self.state.value.lower():
                self.emails_sent += 1
            return f"Transitioned to {self.state.value}"
        return f"No transition for ({self.state.value}, {event})"

# Simulate a prospect journey
seq = SequenceEntry("DataFlow Inc")
events = ["send", "open", "no_reply", "reply", "positive"]
print(f"Simulating sequence for {seq.prospect_company}:\n")
for event in events:
    result = seq.transition(event)
    print(f"  Event: {event:<15s} → State: {seq.state.value}")
print(f"\nFull history:")
for h in seq.history:
    print(f"  {h}")
print(f"\nEmails sent: {seq.emails_sent}")

## Exercises

1. **Add a research confidence score** — Extend the `research_prospect` function to compute a confidence score (0–1) based on data completeness: how many data sources returned results, how recent the news is, how many hooks were found. Score 3 prospects and rank them.

2. **Build a multi-channel sequence** — Extend the `SequenceState` state machine to support LinkedIn connection requests and phone calls as additional touchpoints between emails. Add states like `LINKEDIN_SENT` and `CALL_ATTEMPTED` and the appropriate transitions.

3. **Prompt variant generator** — Create a function that takes one email prompt and generates 3 variations: one with a "resource share" CTA, one with a "quick question" CTA, and one with a "soft meeting" CTA. Score each variant's personalization depth.

## Key Takeaways

| # | Takeaway |
|---|----------|
| 1 | A sales intelligence agent is a pipeline of specialized components with clear data contracts. |
| 2 | Research quality determines email quality — the research agent is the most critical component. |
| 3 | Lead scoring must be explainable; sales teams won't trust a black-box ranking. |
| 4 | Email prompts use a slot-based architecture: research fills personalization slots in the template. |
| 5 | Response classification via embedding centroids works well with few-shot examples and no fine-tuning. |
| 6 | Outreach sequences are state machines — every prospect has a defined path through the campaign. |

## What's Next

- **Next**: [02_build.ipynb](./02_build.ipynb) — Implement the full pipeline from prospect database to end-to-end outreach
- **Related**: [03_evaluate.ipynb](./03_evaluate.ipynb) — Evaluate email quality, classification accuracy, and ROI

## References & Further Reading

1. [Lemkin, J., "The Math Behind Outbound Sales," SaaStr, 2023](https://www.saastr.com/the-math-behind-outbound-sales/) — Unit economics of SDR teams and pipeline generation
2. [Reimers & Gurevych, "Sentence-BERT," EMNLP 2019](https://arxiv.org/abs/1908.10084) — Foundation paper for the embedding model used in response classification
3. [Apollo.io, "The State of Outbound in 2024"](https://www.apollo.io/blog/state-of-outbound) — Industry benchmarks for email sequences and response rates
4. [Salesloft, "Cadence Best Practices," 2024](https://www.salesloft.com/resources/cadence-best-practices) — Multi-touch sequence design patterns